# Bellman-Ford Algorithm

This notebook implements the **Bellman-Ford algorithm** for finding the shortest paths from a single source vertex to all other vertices in a weighted, directed graph.

Unlike Dijkstra's algorithm, Bellman-Ford can handle graphs with **negative edge weights**, and it can also **detect negative weight cycles** — cases where the shortest path is undefined because a loop keeps decreasing the total distance.

**Contents:**
1. Build a `bellman_ford` function that relaxes all edges `n-1` times, then checks for negative cycles.
2. Build a helper `init` function that converts an adjacency matrix into an edge list.
3. Define a sample 10-node graph as an adjacency matrix.
4. Run the algorithm from source node `0` and inspect the shortest distances.

In [ ]:
import numpy as np

## Step 1: The Bellman-Ford Function

`bellman_ford(n, edges, src)` takes:
- `n` — the number of vertices
- `edges` — a list of `(weight, u, v)` tuples representing directed edges from `u` to `v`
- `src` — the source vertex to compute shortest paths from

**How it works:**
1. Initialize all distances to infinity, except the source (distance `0`).
2. **Relax** every edge `n-1` times: for each edge `(w, u, v)`, if going through `u` gives a shorter path to `v`, update `dist[v]`. Repeating this `n-1` times guarantees convergence, since the longest possible shortest path uses at most `n-1` edges.
3. Do one more pass over all edges. If any distance can still be improved, a **negative weight cycle** exists, and the function reports it and returns `None`.
4. Otherwise, return the array of shortest distances from `src` to every vertex.

In [ ]:
def bellman_ford(n,edges,src):
    dist = np.full(n,np.inf)
    dist[src] = 0
    for i in range(n-1):
        for w,u,v in edges:
            if(dist[v] > dist[u] + w):
                dist[v] = dist[u] + w

    for w,u,v in edges:
                if(dist[v] > dist[u] + w):
                   print("Negative weight Cycle detected")
                   return None
    return dist    

## Step 2: Converting an Adjacency Matrix to an Edge List

`bellman_ford` expects a list of `(weight, u, v)` edges, but graphs are often easier to define as an **adjacency matrix**. The `init(graph)` function bridges the two representations.

- `graph[i][j]` holds the weight of the edge from vertex `i` to vertex `j`, or `np.inf` if no edge exists.
- The function scans the **upper triangle** of the matrix (`j > i`) and collects every existing edge as a `(weight, i, j)` tuple.
- It returns both `n` (the number of vertices, from the matrix size) and the resulting `edges` list.

**Note:** Because only the upper triangle (`i < j`) is scanned, edges are effectively treated as going from a lower-indexed vertex to a higher-indexed one. Any weight placed in the lower triangle (like `graph[9][0]`) is skipped by this function — this matters for the example graph below.

In [ ]:
def init(graph):
    edges = []
    n = len(graph)
    for i in range(n):
        for j in range(i + 1, n):
            if graph[i][j] != np.inf:
                edges.append((graph[i][j],i,j))
    return n,edges

## Step 3: Example Graph

Below is a 10-node graph (indices `0`–`9`) encoded as a 10×10 adjacency matrix, with `np.inf` marking "no direct edge."

For example:
- Node `0` connects to node `1` (weight `4`) and node `4` (weight `2`).
- Node `9` connects back to node `0` (weight `1`), forming a cycle through the graph.

Since `init()` only reads the **upper triangle** of the matrix, the edge `graph[9][0] = 1` (in the lower triangle) will **not** be picked up as a usable edge in this run — worth keeping in mind when interpreting the results below.

In [ ]:
graph = np.array([
    [0,   4,   np.inf, np.inf, 2,   np.inf, np.inf, np.inf, np.inf, np.inf],
    [np.inf, 0,  3,   np.inf, np.inf, 5,   np.inf, np.inf, np.inf, np.inf],
    [np.inf, np.inf, 0,   2,   np.inf, np.inf, 4,   np.inf, np.inf, np.inf],
    [np.inf, np.inf, np.inf, 0,   1,   np.inf, np.inf, 6,   np.inf, np.inf],
    [np.inf, np.inf, np.inf, np.inf, 0,   3,   np.inf, np.inf, 7,   np.inf],
    [np.inf, np.inf, np.inf, np.inf, np.inf, 0,   2,   np.inf, np.inf, 8],
    [np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, 0,   1,   np.inf, np.inf],
    [np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, 0,   3,   np.inf],
    [np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, 0,   2],
    [1,   np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, 0]
], dtype=float)

## Step 4: Running the Algorithm

Now we convert the graph into an edge list and run Bellman-Ford from source vertex `0`.

In [ ]:
n,edges = init(graph)

print(bellman_ford(n,edges,0))

[ 0.  4.  7.  9.  2.  5.  7.  8.  9. 11.]


## Results

The output array gives the shortest distance from node `0` to every other node:

| Node | Distance |
|------|----------|
| 0    | 0  |
| 1    | 4  |
| 2    | 7  |
| 3    | 9  |
| 4    | 2  |
| 5    | 5  |
| 6    | 7  |
| 7    | 8  |
| 8    | 9  |
| 9    | 11 |

**Sanity checks:**
- `dist[0] = 0`, as expected for the source itself.
- `dist[4] = 2` matches the direct edge `0 → 4`.
- `dist[1] = 4` matches the direct edge `0 → 1`; note there's no shorter path via node `4`, since `4 → 1` isn't an edge in this graph.
- No "Negative weight Cycle detected" message was printed, confirming the graph (as read by `init`, i.e. only its upper-triangle edges) contains no negative cycles.

Since edge `graph[9][0]` in the lower triangle was excluded by `init()`, no cycle was ever formed in the edge list actually used here — so that check didn't have anything to catch. If you want the algorithm to also consider edges below the diagonal (e.g. to test true cycle detection), `init()` would need to scan the full matrix rather than just `j > i`.